# Platform Monitoring Queries

This notebook answers questions by analyzing pipeline execution metadata stored in the `pipeline_executions` Delta table. 

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("AnalyzePipelineMonitoring")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

PIPELINE_EXECUTIONS_PATH = (
    "s3a://delta-data/metadata/pipeline_executions"
)

executions = (
    spark.read
    .format("delta")
    .load(PIPELINE_EXECUTIONS_PATH)
)

executions.createOrReplaceTempView("pipeline_executions")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 16:07:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 16:07:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Questions

### Which dataset fails validation most frequently?

In [2]:
spark.sql("""
    SELECT
        dataset,
        SUM(
            CASE
                WHEN COALESCE(validation_failure_count, 0) > 0
                THEN 1
                ELSE 0
            END
        ) AS executions_with_failures,
        SUM(
            COALESCE(validation_failure_count, 0)
        ) AS total_failed_rules
    FROM pipeline_executions
    WHERE stage = 'silver'
    GROUP BY dataset
    ORDER BY total_failed_rules DESC, executions_with_failures DESC
""").show(truncate=False)

26/09/24 16:21:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------------------+------------------+
|dataset    |executions_with_failures|total_failed_rules|
+-----------+------------------------+------------------+
|taxi_trips |2                       |30                |
|taxi_zones |2                       |4                 |
|air_quality|0                       |0                 |
|weather    |0                       |0                 |
+-----------+------------------------+------------------+



### Which dataset requires the longest processing time?

In [3]:
spark.sql("""
    SELECT
        stage,
        dataset,
        COUNT(*) AS execution_count,
        ROUND(AVG(duration_seconds), 3) AS average_duration_seconds,
        ROUND(MAX(duration_seconds), 3) AS longest_duration_seconds
    FROM pipeline_executions
    WHERE status = 'SUCCESS'
    GROUP BY stage, dataset
    ORDER BY average_duration_seconds DESC
""").show(truncate=False)

+-----------------+--------------------------------+---------------+------------------------+------------------------+
|stage            |dataset                         |execution_count|average_duration_seconds|longest_duration_seconds|
+-----------------+--------------------------------+---------------+------------------------+------------------------+
|bronze           |taxi_zones                      |4              |53.105                  |181.315                 |
|silver           |air_quality                     |5              |50.256                  |105.289                 |
|gold             |integrated_taxi_trips           |3              |41.678                  |60.808                  |
|silver           |taxi_trips                      |5              |41.182                  |64.818                  |
|bronze           |air_quality                     |4              |12.445                  |14.213                  |
|bronze           |taxi_trips                   

### How many records were rejected during each execution?

In [11]:
spark.sql("""
    SELECT
        pipeline_run_id,
        execution_id,
        stage,
        dataset,
        rejected_records,
        started_at
    FROM pipeline_executions
    ORDER BY rejected_records DESC, started_at DESC
""").show(100, truncate=False)

+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|pipeline_run_id                     |execution_id                        |stage            |dataset                         |rejected_records|started_at                |
+------------------------------------+------------------------------------+-----------------+--------------------------------+----------------+--------------------------+
|716a99a3-556c-48aa-b434-8a653b4a0dd8|3c132477-6466-404b-b109-67dde2cc89b0|silver           |taxi_trips                      |136945          |2026-09-24 15:45:22.64244 |
|c8869316-25aa-4fd7-96b9-260afc06891c|67f28a24-0c38-4bb0-b9f3-b8536a651033|silver           |taxi_trips                      |136945          |2026-09-24 15:24:43.784774|
|9b2b2d79-60d1-4df0-904f-5dc385941670|676d3bf9-3da3-4038-b88f-320419eccf2f|silver           |taxi_trips                      |136945          |20

### How has processing time changed over multiple executions?

In [10]:
spark.sql("""
    SELECT
        stage,
        dataset,
        pipeline_run_id,
        started_at,
        duration_seconds
    FROM pipeline_executions
    WHERE status = 'SUCCESS'
    ORDER BY stage, dataset, started_at
""").show(100, truncate=False)

+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|stage            |dataset                         |pipeline_run_id                     |started_at                |duration_seconds|
+-----------------+--------------------------------+------------------------------------+--------------------------+----------------+
|analytics_refresh|air_quality_vs_demand           |716a99a3-556c-48aa-b434-8a653b4a0dd8|2026-09-24 15:47:18.394373|2.255           |
|analytics_refresh|average_trip_distance_by_weather|716a99a3-556c-48aa-b434-8a653b4a0dd8|2026-09-24 15:47:14.636043|2.709           |
|analytics_refresh|monthly_zone_demand             |716a99a3-556c-48aa-b434-8a653b4a0dd8|2026-09-24 15:47:10.711883|3.745           |
|analytics_refresh|zone_weather_demand_variation   |716a99a3-556c-48aa-b434-8a653b4a0dd8|2026-09-24 15:47:20.819129|2.004           |
|bronze           |air_quality                     |e8e98001-2